# Testing the effect of population vs. individual thermal noise (SI)

In [3]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import scipy
import pandas as pd
import random
import sys
from matplotlib import cm
import matplotlib as mpl
sys.path.insert(1, '../scripts/03_analytical_prediction')
from tpc_functions_oo import *
tpc_object = tpc_functions()
import tskit
from matplotlib.lines import Line2D
datadir = "../data/"
plt.rcParams.update({'font.size': 15})

from pathlib import Path



## 0. Set up

Proportion of shared thermal variance ($p = \sigma_{pop}^2 / \sigma^2$) was scanned between 0 and 1 with 0.1 increment. For each proportion, I simulate for 20,000 generations (or until population crashes because everyone's fitness is zero), and repeat it 30 times with seed = 0 to 29. 

## 1. Duration of the simulation
All simulations completed without extinction for $p= 0, 0.1, 0.2, 0.3, 0.4$, but only 18 (out of 30) completed for $p=0.5$. All simulations for $p > 0.5$ crashed. This makes sense since big thermal noise that is shared for everyone puts the entire population in danger. Duration of simulation is summarized in the first table (Also in SI)

## 2. Distribution of evolved TPCs (only for survived populations)
The greater $p$ is, more TPCs evolve extremely high $CTmax$. However, there isn't a noticeable different in genetic or phenotypic correlation between $B$ and $CTmin$. Both tables are in the SI of the manuscript as well.

In [4]:
params = pd.read_csv("../scripts/01_prepare_input_parameters/include_sd_pop_params.csv")
params_sub = params[params.seed==29]


In [39]:
def duration(row):
    try:
        log = pd.read_csv(f"{datadir}{row.OUTNAME}.txt")
        final_cycle = log.cycle.iloc[-1] - row.BURNIN
    except pd.errors.EmptyDataError:
        final_cycle = pd.NA
    return (np.round(row.STDEV_TEMP_POP ** 2 / 100, 1), row.seed, final_cycle)

duration_df = params.apply(duration, axis=1, result_type='expand')
duration_df.columns = ['var_pop_ratio', 'seed', 'post_burnin_duration']
summary_df = duration_df.dropna().groupby(['var_pop_ratio']).agg({
    'post_burnin_duration': ['min', 'mean', 'median', 'max'],
})
print(summary_df.to_latex())

\begin{tabular}{lllll}
\toprule
 & \multicolumn{4}{r}{post_burnin_duration} \\
 & min & mean & median & max \\
var_pop_ratio &  &  &  &  \\
\midrule
0.000000 & 15000.000000 & 15000.000000 & 15000.000000 & 15000.000000 \\
0.100000 & 15000.000000 & 15000.000000 & 15000.000000 & 15000.000000 \\
0.200000 & 15000.000000 & 15000.000000 & 15000.000000 & 15000.000000 \\
0.300000 & 15000.000000 & 15000.000000 & 15000.000000 & 15000.000000 \\
0.400000 & 15000.000000 & 15000.000000 & 15000.000000 & 15000.000000 \\
0.500000 & 1920.000000 & 13121.466667 & 15000.000000 & 15000.000000 \\
0.600000 & 74.000000 & 3636.666667 & 2641.500000 & 12530.000000 \\
0.700000 & 8.000000 & 430.344828 & 341.000000 & 1592.000000 \\
0.800000 & 5.000000 & 87.448276 & 81.000000 & 351.000000 \\
0.900000 & 1.000000 & 19.888889 & 15.000000 & 116.000000 \\
1.000000 & 1.000000 & 5.043478 & 2.000000 & 29.000000 \\
\bottomrule
\end{tabular}



In [32]:
def classify_final_TPCs(row):
    # threshold w_component when parameter = parameter_critical + Delta_parameter (replace parameter with CTmin, CTmax or B)
    threshold = 0.5
    # First check if csv file is empty
    try:
        B_and_CTmin = pd.read_csv(f'{datadir}{row.OUTNAME}.csv', engine='python', sep=',')
        if B_and_CTmin.empty:
            print("The CSV file contains a header but no data.")
            return (np.round(row.STDEV_TEMP_POP ** 2 / 100, 1), row.seed, pd.NA, pd.NA, pd.NA, pd.NA, pd.NA)
        else:
            print("The CSV file has data.")
            Bs_final = np.array(B_and_CTmin.B)
            CTmins_final = np.array(B_and_CTmin.CTmin)

            n_green = 0
            n_blue = 0
            n_red = 0
            n_grey = 0
            n_others = 0
            for B, CTmin in zip(Bs_final, CTmins_final):
                w_CTmin = tpc_object.w_CTmin(CTmin=CTmin)
                w_B = tpc_object.w_B(B=B)
                CTmax = CTmin + B
                w_CTmax = tpc_object.w_CTmax(CTmax=CTmax)
                if (w_B <= threshold) and (w_CTmin > threshold) and (w_CTmax > threshold):
                    # Too generalist
                    n_green += 1
                elif (w_CTmin <= threshold) and (w_B > threshold) and (w_CTmax > threshold):
                    # Too much cold adaptation
                    n_blue += 1
                elif (w_B > threshold) and (w_CTmin > threshold) and (w_CTmax <= threshold):
                    # Too much heat adaptation
                    n_red += 1
                elif (np.array([w_B, w_CTmin, w_CTmax]) > threshold).all():
                    # not limited by any of 3 physiological constraints
                    n_grey += 1
                else:
                    # other kinds
                    print(f"mean temperature={row.MEAN_TEMP}, std={row.STDEV_TEMP}")
                    print(f"w_B={w_B}, w_CTmin={w_CTmin}, w_CTmax={w_CTmax}")
                    n_others += 1
            n_total = n_green + n_blue + n_red + n_grey + n_others
            ratio_green = n_green / n_total
            ratio_blue = n_blue / n_total
            ratio_red = n_red / n_total
            ratio_grey = n_grey / n_total
            ratio_others = n_others / n_total
            return (np.round(row.STDEV_TEMP_POP ** 2 / 100, 1), row.seed, ratio_green, ratio_blue, ratio_red, ratio_grey, ratio_others)

    except pd.errors.EmptyDataError:
        print("The CSV file is completely empty (no header or data).")


    

final_tpc_classified_df = params.apply(classify_final_TPCs, axis=1, result_type='expand')
final_tpc_classified_df.columns = ['var_pop_ratio', 'seed', 'ratio_green', 'ratio_blue', 'ratio_red', 'ratio_grey', 'ratio_others']
print(final_tpc_classified_df.tail())

The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file has data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file contains a header but no data.
The CSV file con

In [33]:
summary_df = final_tpc_classified_df.dropna().groupby(['var_pop_ratio']).agg({
    'ratio_red': ['mean', 'std'],
})
print(summary_df.to_latex())


\begin{tabular}{llr}
\toprule
 & \multicolumn{2}{r}{ratio_red} \\
 & mean & std \\
var_pop_ratio &  &  \\
\midrule
0.000000 & 0.056274 & 0.004145 \\
0.100000 & 0.057805 & 0.003736 \\
0.200000 & 0.058954 & 0.004668 \\
0.300000 & 0.060561 & 0.004055 \\
0.400000 & 0.062736 & 0.003941 \\
0.500000 & 0.069361 & 0.006389 \\
\bottomrule
\end{tabular}



In [34]:
def genetic_phenotypic_corr(row):
    # Calculate r_g and r_p for row, if tree sequence exists (i.e. if simulation didn't crash)
    # threshold w_component when parameter = parameter_critical + Delta_parameter (replace parameter with CTmin, CTmax or B)
    ts_path = Path(f'{datadir}{row.OUTNAME}.trees')
    if ts_path.is_file():
        B_and_CTmin = pd.read_csv(f'{datadir}{row.OUTNAME}.csv', engine='python', sep=',')
        Bs_final = np.array(B_and_CTmin.B)
        CTmins_final = np.array(B_and_CTmin.CTmin)
        p_corr = np.corrcoef(Bs_final, CTmins_final)[0,1]

        ts = tskit.load(ts_path)
        # Calculate genetic covariance (sum of effect sizes in QTN_CTmin vs. QTN_B)
        G_CTmin_list = np.zeros(ts.num_individuals)
        G_B_list = np.zeros(ts.num_individuals)
        
        var_idx = 0
        for var in ts.variants():
            gene_dose = var.genotypes[::2] + var.genotypes[1::2]
            s = var.site.mutations[0].metadata['mutation_list'][0]['selection_coeff']
            if var.site.mutations[0].metadata['mutation_list'][0]['mutation_type'] == 2:
            # m2, i.e. QTN for B
                G_B_list += gene_dose * s
            elif var.site.mutations[0].metadata['mutation_list'][0]['mutation_type'] == 3:
                G_CTmin_list += gene_dose * s
            else:
                print("undefined mutation type")
            var_idx += 1

        g_corr = np.corrcoef(G_B_list, G_CTmin_list)[0,1]
        
    else:
        print("simulation crashed, no tree-sequence to find correlation from")
        g_corr = pd.NA
        p_corr = pd.NA
    return (np.round(row.STDEV_TEMP_POP ** 2 / 100, 1), row.seed, g_corr, p_corr)
    

corr_df = params.apply(genetic_phenotypic_corr, axis=1, result_type='expand')
corr_df.columns = ['var_pop_ratio', 'seed', 'g_corr', 'p_corr']
print(corr_df.head())

simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulation crashed, no tree-sequence to find correlation from
simulati

In [35]:
summary_df = corr_df.dropna().groupby(['var_pop_ratio']).agg({
    'g_corr': ['mean', 'std'],
    'p_corr': ['mean', 'std'],
})
print(summary_df.to_latex())

\begin{tabular}{llrlr}
\toprule
 & \multicolumn{2}{r}{g_corr} & \multicolumn{2}{r}{p_corr} \\
 & mean & std & mean & std \\
var_pop_ratio &  &  &  &  \\
\midrule
0.000000 & -0.212202 & 0.155904 & -0.029232 & 0.027262 \\
0.100000 & -0.240603 & 0.143441 & -0.032536 & 0.020201 \\
0.200000 & -0.273171 & 0.155509 & -0.036586 & 0.028900 \\
0.300000 & -0.234148 & 0.136214 & -0.032899 & 0.026535 \\
0.400000 & -0.229996 & 0.140913 & -0.028610 & 0.028210 \\
0.500000 & -0.233980 & 0.128824 & -0.029069 & 0.024612 \\
\bottomrule
\end{tabular}

